In [1]:
import os
import ee
import math
import time
import geemap
import datetime 
import pandas as pd

import yaml

with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

ee.Authenticate(force=True)

# Initialize Earth Engine
try:
    ee.Initialize(project=config['gee']['project_id'], opt_url='https://earthengine-highvolume.googleapis.com')
    print('Google Earth Engine initialized successfully!')
except ee.EEException as e:
    print('Google Earth Engine failed to initialize!', e)
    raise

/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Enter verification code:  4/1AfrIepAsPE_YUNKjFL7Td5rjaousyrD5wfThXmlONKXjowfJ08w2kSk1NRY



Successfully saved authorization token.
Google Earth Engine initialized successfully!


In [2]:
# sorted(os.listdir("./shc_data/NORMALISED_DATA/2024"))
sorted(os.listdir("./shc_data/NORMALISED_DATA/AGRI_2023-24/"))

['ANDAMAN & NICOBAR_AGRI_2023-24.csv',
 'ANDHRA PRADESH_AGRI_2023-24.csv',
 'ARUNACHAL PRADESH_AGRI_2023-24.csv',
 'ASSAM_AGRI_2023-24.csv',
 'BIHAR_AGRI_2023-24.csv',
 'CHHATTISGARH_AGRI_2023-24.csv',
 'GOA_AGRI_2023-24.csv',
 'GUJARAT_AGRI_2023-24.csv',
 'HARYANA_AGRI_2023-24.csv',
 'HIMACHAL PRADESH_AGRI_2023-24.csv',
 'JAMMU & KASHMIR_AGRI_2023-24.csv',
 'JHARKHAND_AGRI_2023-24.csv',
 'KARNATAKA_AGRI_2023-24.csv',
 'LADAKH_AGRI_2023-24.csv',
 'MADHYA PRADESH_AGRI_2023-24.csv',
 'MAHARASHTRA_AGRI_2023-24.csv',
 'MEGHALAYA_AGRI_2023-24.csv',
 'MIZORAM_AGRI_2023-24.csv',
 'NAGALAND_AGRI_2023-24.csv',
 'ODISHA_AGRI_2023-24.csv',
 'PUDUCHERRY_AGRI_2023-24.csv',
 'PUNJAB_AGRI_2023-24.csv',
 'RAJASTHAN_AGRI_2023-24.csv',
 'SIKKIM_AGRI_2023-24.csv',
 'TAMIL NADU_AGRI_2023-24.csv',
 'TELANGANA_AGRI_2023-24.csv',
 'TRIPURA_AGRI_2023-24.csv',
 'UTTAR PRADESH_AGRI_2023-24.csv',
 'UTTARAKHAND_AGRI_2023-24.csv',
 'WEST BENGAL_AGRI_2023-24.csv']

## Please use the same names as listed below for the state variable

0: 
Andaman & Nicobar
1: 
Andhra Pradesh
2: 
Arunachal Pradesh
3: 
Assam
4: 
Bihar
5: 
Chandigarh
6: 
Chhattishgarh
7: 
Daman and Diu and Dadra and Nagar Haveli
8: 
Delhi
9: 
Goa
10: 
Gujarat
11: 
Haryana
12: 
Himachal Pradesh
13: 
Jammu and Kashmir
14: 
Jharkhand
15: 
Karnataka
16: 
Kerala
17: 
Ladakh
18: 
Lakshadweep
19: 
Madhya Pradesh
20: 
Maharashtra
21: 
Manipur
22: 
Meghalaya
23: 
Mizoram
24: 
Nagaland
25: 
Odisha
26: 
Puducherry
27: 
Puducherry
28: 
Punjab
29: 
Rajasthan
30: 
Sikkim
31: 
Tamilnadu
32: 
Telengana
33: 
Tripura
34: 
Uttar Pradesh
35: 
Uttarakhand
36: 
West Bengal

In [95]:
# === CONFIGURABLE PARAMETERS ===
# DATA_DIR = "./shc_data/NORMALISED_DATA/AGRI_2023-24/"
DATA_DIR = os.path.join(
    config['paths']['data_dir'], 
    config['paths']['output_dir'], 
    config['paths']['output_sub_dir']
)
INPUT_CSV = os.path.join(DATA_DIR, "NAGALAND_AGRI_2023-24.csv")  # CSV containing soil properties
BATCH_SIZE = 3000  # Number of points processed per batch



# india_districts = ee.FeatureCollection("projects/ee-mtpictd-aakash-mcs/assets/state")
# state = india_districts.filter(ee.Filter.eq('State_Name', 'Karnataka'))

# # Load the public dataset for first-level administrative boundaries (states/provinces)
# admin_boundaries = ee.FeatureCollection("FAO/GAUL/2015/level1")

# # karnataka, tamil nadu, kerala, goa, maharashtra, andhra pradesh, telangana
# # Filter for India first, then for the specific state
# india_boundaries = admin_boundaries.filter(ee.Filter.eq('ADM0_NAME', 'India'))
# state = india_boundaries.filter(ee.Filter.eq('ADM1_NAME', 'Arunachal Pradesh')) # Karnataka Tamil Nadu Puducherry Andhra Pradesh Goa Maharashtra Telengana

# india_districts = ee.FeatureCollection("projects/ee-mtpictd-aakash-mcs/assets/state")
# Load the asset path and state filter from config
india_districts = ee.FeatureCollection(config['gee']['state_asset_path'])
state = india_districts.filter(ee.Filter.eq('State_Name', 'Nagaland'))

# Initialize start date and end date
# SD = pd.to_datetime("2023-07-01")
# ED = pd.to_datetime("2024-06-30")
# Initialize start date and end date from config
SD = pd.to_datetime(config['parameters']['start_date'])
ED = pd.to_datetime(config['parameters']['end_date'])

In [96]:
def maskS2clouds(image):
    qa = image.select('QA60');
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloudBitMask = 1 << 10;
    cirrusBitMask = 1 << 11;
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    scaled = image.divide(10000)
    scaled = scaled.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'], ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2'])
    return scaled.updateMask(mask).toFloat()



def fetch_satellite_data(start_date, end_date, roi):
    """Fetches physical properties, Landsat 8/9 SR annual stack, and seasonal NDVI bands."""

    # ----- Landsat 8/9 SR: mask clouds/shadows, scale reflectance, rename bands -----
    def mask_scale_rename_ls(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)          # bit 3: clouds (keep clear)
        cloud_shad = qa.bitwiseAnd(1 << 4).eq(0)     # bit 4: cloud shadow (keep clear)
        cirrus = qa.bitwiseAnd(1 << 2).eq(0)         # bit 2: cirrus (keep clear)
        snow = qa.bitwiseAnd(1 << 5).eq(0)           # bit 5: snow (keep clear)
        mask = cloud.And(cloud_shad).And(cirrus).And(snow)
        # Scale to reflectance: SR = DN * 0.0000275 - 0.2 (Collection 2 Level-2)
        scaled = image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']) \
                      .multiply(0.0000275).add(-0.2)
        # Rename to match your feature names
        scaled = scaled.rename(['BLUE','GREEN','RED','NIR','SWIR1','SWIR2'])
        # Add NDVI band (using renamed NIR/RED)
        ndvi = scaled.normalizedDifference(['NIR','RED']).rename('NDVI')
        return image.addBands(scaled, overwrite=True) \
                    .addBands(ndvi) \
                    .updateMask(mask)

    # Landsat 8/9 C2 L2 SR collections
    L8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    L9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    LS = L8.merge(L9) \
           .filterBounds(roi) \
           .filterDate(start_date, end_date) \
           .map(mask_scale_rename_ls)

    # ----- MODIS LST (scaled), CHIRPS precip -----
    def scale_lst(img): return img.multiply(0.002)
    temp = (ee.ImageCollection('MODIS/061/MOD11A2')
            .select('LST_Day_1km')
            .filterBounds(roi).filterDate(start_date, end_date)
            .map(scale_lst).mean().rename('temp'))
    preci = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
             .filterBounds(roi).filterDate(start_date, end_date)
             .mean().rename('precipitation'))

    # ----- Terrain and SoilGrids -----
    elevation = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(roi)
    slope = ee.Terrain.slope(elevation).rename('slope')
    aspect = ee.Terrain.aspect(elevation).rename('aspect')
    slope_radians = slope.multiply(math.pi).divide(180)
    flow_acc = ee.Image('MERIT/Hydro/v1_0_1').select('upa').clip(roi)
    tan_slope = slope_radians.tan()
    safe_slope = tan_slope.where(tan_slope.eq(0), 0.001)
    twi = flow_acc.divide(safe_slope).log().rename('TWI')
    sand = ee.Image('projects/soilgrids-isric/sand_mean') \
              .select(['sand_0-5cm_mean','sand_5-15cm_mean']).clip(roi) \
              .rename(['sand05','sand515'])
    silt = ee.Image('projects/soilgrids-isric/silt_mean') \
              .select(['silt_0-5cm_mean','silt_5-15cm_mean']).clip(roi) \
              .rename(['silt05','silt515'])
    clay = ee.Image('projects/soilgrids-isric/clay_mean') \
              .select(['clay_0-5cm_mean','clay_5-15cm_mean']).clip(roi) \
              .rename(['clay05','clay515'])

    # ----- Annual Landsat spectral stack (mean) -----
    ls_annual = LS.select(['BLUE','GREEN','RED','NIR','SWIR1','SWIR2']).mean()

    # ----- Seasonal NDVI from Landsat (median suggested) -----
    kharif = LS.filterDate('2023-07-01','2023-10-31').select('NDVI')
    rabi   = LS.filterDate('2023-11-01','2024-03-31').select('NDVI')
    zaid   = LS.filterDate('2024-04-01','2024-06-30').select('NDVI')

    def reduce_season(season_ic, name):
        return ee.Image(ee.Algorithms.If(
            season_ic.size().gt(0),
            season_ic.median().rename(name).unmask(0),
            ee.Image.constant(0).toFloat().rename(name)
        ))

    ndvi_kharif = reduce_season(kharif, 'NDVI_Kharif')
    ndvi_rabi   = reduce_season(rabi,   'NDVI_Rabi')
    ndvi_zaid   = reduce_season(zaid,   'NDVI_Zaid')

    # ----- Build final image with addBands on a base image to ensure compatibility -----
    collection = elevation.addBands([
        ls_annual, temp, preci, slope, aspect, twi, sand, silt, clay,
        ndvi_kharif, ndvi_rabi, ndvi_zaid
    ]).clip(roi).toFloat()

    return collection




def fetch_indices_for_district(df, district, month, start_date, end_date):
    """Processes a district's data in batches and fetches satellite indices."""
    
    # Filter district data
    district_df = df[df["district"] == district].reset_index(drop=True)
    total_features = len(district_df)
    
    processed = 0
    batch_number = 1

    while processed < total_features:
        batch_df = district_df.iloc[processed : processed + BATCH_SIZE]
        valid_points = []
        # loop through the district dataframe batch
        for _, row in batch_df.iterrows():
            # sanity check if survey data lies between start and end date
            if (pd.to_datetime(row['date']) >= SD and pd.to_datetime(row['date']) <= ED):
                valid_points.append(
                    ee.Feature(ee.Geometry.Point(row['long'], row['lat']), {
                        'district' : row['district'],
                        'village' : row['village'],
                        'date': row['date'],
                        'start_date': SD,
                        'end_date': ED,
                        'N': row['N'],
                        'P': row['P'],
                        'K': row['K'],
                        'B': row['B'],
                        'Fe': row['Fe'],
                        'Zn': row['Zn'],
                        'Cu': row['Cu'],
                        'S': row['S'],
                        'OC': row['OC'],
                        'pH': row['pH'],
                        'Mn': row['Mn'],
                        'EC': row['EC']}))

        # --- ADD THIS DEBUGGING BLOCK ---
        print(f"--- Processing Batch {batch_number} for district {district} ---")
        print(f"Found {len(valid_points)} valid points in this batch.")

        if not valid_points:
            print("WARNING: No valid points found in this batch after date filtering. Skipping to next batch.")
            processed += BATCH_SIZE
            batch_number += 1
            continue  # This skips the rest of the loop for this empty batch
        # --- END DEBUGGING BLOCK ---

        points = ee.FeatureCollection(valid_points) 
        # print(f"Points before state geometry filter: {points.size().getInfo()}")
        points = points.filterBounds(state.geometry())
        # print(f"Points after state geometry filter: {points.size().getInfo()}")
        roi = points.geometry().bounds()
        collections = fetch_satellite_data(start_date, end_date, roi)
        sampled_points = collections.sampleRegions(
            collection=points, scale=30, tileScale=8, geometries=True
        )

        state_name = INPUT_CSV.split("/")[-1].split(".csv")[0].replace(" ", "_").replace("&", "_")

        SELECTORS = ['district','village','date', 'start_date','end_date', 'N','P','K','B','Fe','Zn','Cu','S','OC','pH','Mn','EC','BLUE','GREEN','RED','NIR','SWIR1','SWIR2','temp','precipitation','elevation','slope','aspect','TWI','sand05','sand515','silt05','silt515','clay05','clay515','NDVI_Kharif','NDVI_Rabi','NDVI_Zaid', '.geo']
        # Export the CSV file to google drive
        task = ee.batch.Export.table.toDrive(
            collection=sampled_points,
            description=f"{state_name}_sampled_batch_{batch_number}",
            folder='GEE_Exports_Ratinder',
            fileNamePrefix=f"{state_name}_{district}_batch{batch_number}",
            fileFormat='CSV',
            selectors=SELECTORS
        )
        task.start()

        processed += BATCH_SIZE
        batch_number += 1
        print(f"Batch {batch_number-1} processed. ")

In [97]:
def main():
    """Main function to process all districts."""
    df = pd.read_csv(INPUT_CSV)
    
    # Ensure necessary columns exist
    required_columns = {"long", "lat", "district", "village", "date"}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"CSV file must contain columns: {required_columns}")

    # Process districts
    unique_districts = df["district"].unique()
    print(f"Total Districts: {len(unique_districts)}")

    start_date, end_date = ee.Date(SD), ee.Date(ED)
    
    for district in unique_districts:
        start_time = time.perf_counter()

        print(f"\nProcessing District: {district}")
        batch_df = df[df["district"] == district].reset_index(drop=True)
        total_features = len(batch_df)
        print(f"Total Features in {district}: {total_features}")
            
        fetch_indices_for_district(df, district, None, start_date, end_date)
            
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time
        a = datetime.timedelta(seconds=elapsed_time)
        print("Time taken : " + str(a))
        
    print("\nAll districts processed successfully!")

In [98]:
main()

Total Districts: 16

Processing District: CHUMOUKEDIMA
Total Features in CHUMOUKEDIMA: 160
--- Processing Batch 1 for district CHUMOUKEDIMA ---
Found 160 valid points in this batch.
Batch 1 processed. 
Time taken : 0:00:01.670087

Processing District: DIMAPUR
Total Features in DIMAPUR: 1077
--- Processing Batch 1 for district DIMAPUR ---
Found 1077 valid points in this batch.
Batch 1 processed. 
Time taken : 0:00:06.529837

Processing District: KIPHIRE
Total Features in KIPHIRE: 1924
--- Processing Batch 1 for district KIPHIRE ---
Found 1924 valid points in this batch.
Batch 1 processed. 
Time taken : 0:00:10.336553

Processing District: KOHIMA
Total Features in KOHIMA: 119
--- Processing Batch 1 for district KOHIMA ---
Found 119 valid points in this batch.
Batch 1 processed. 
Time taken : 0:00:01.375534

Processing District: LONGLENG
Total Features in LONGLENG: 328
--- Processing Batch 1 for district LONGLENG ---
Found 328 valid points in this batch.
Batch 1 processed. 
Time taken : 0